# Real Adapter Fine-Tuning with DistilBERT + IMDb

In this notebook, we use a **real pretrained Transformer** (`distilbert-base-uncased`) and the IMDb movie-review dataset.

Goal: predict whether a movie review is **Positive** or **Negative**.

What we will do:
1. Load a pretrained DistilBERT model and its tokenizer.
2. Freeze the original DistilBERT weights.
3. Insert a small trainable adapter after each Transformer block.
4. Train the adapters and a new classification head on IMDb.
5. Evaluate the model and predict sentiment for your own sentences.

**Important:** The notebook defaults to a small training subset so it is practical on Google Colab. You can increase the sample limits later. The pretrained model and dataset are downloaded from Hugging Face, so internet access is required.

## 1. Install libraries

Run this cell once in Google Colab or a fresh notebook environment.

In [15]:
!pip -q install -U transformers datasets accelerate scikit-learn

## 2. Import libraries and choose device

In [16]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, DataCollatorWithPadding
from sklearn.metrics import accuracy_score
import numpy as np

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


## 3. Set the model and training settings

`MAX_TRAIN_SAMPLES` and `MAX_TEST_SAMPLES` limit the dataset size to make the first run faster. Set either to `None` to use the full split.

A Colab T4 GPU is recommended. Start with the defaults before increasing the dataset.

In [17]:
MODEL_NAME = 'distilbert-base-uncased'
MAX_LENGTH = 256
BATCH_SIZE = 16
EPOCHS = 2
LEARNING_RATE = 2e-4
BOTTLENECK_SIZE = 64

MAX_TRAIN_SAMPLES = 5000   # Set to None for all 25,000 training reviews
MAX_TEST_SAMPLES = 2000    # Set to None for all 25,000 test reviews

print('Model:', MODEL_NAME)
print('Train sample limit:', MAX_TRAIN_SAMPLES)
print('Test sample limit:', MAX_TEST_SAMPLES)

Model: distilbert-base-uncased
Train sample limit: 5000
Test sample limit: 2000


## 4. Load the IMDb dataset

IMDb has movie reviews with labels:
- `0` = Negative
- `1` = Positive

We select a smaller subset for the first practice run.

In [18]:
dataset = load_dataset("stanfordnlp/imdb")

train_dataset = dataset['train'].shuffle(seed=42)
test_dataset = dataset['test'].shuffle(seed=42)

if MAX_TRAIN_SAMPLES is not None:
    train_dataset = train_dataset.select(range(min(MAX_TRAIN_SAMPLES, len(train_dataset))))

if MAX_TEST_SAMPLES is not None:
    test_dataset = test_dataset.select(range(min(MAX_TEST_SAMPLES, len(test_dataset))))

print('Training reviews:', len(train_dataset))
print('Testing reviews:', len(test_dataset))
print('Example review:', train_dataset[0]['text'][:400])
print('Example label:', train_dataset[0]['label'])

Training reviews: 5000
Testing reviews: 2000
Example review: There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. Fortier's plot are far more complicated... Fortier looks more like Prime Suspect, if we have to spot similarities... The main character is weak and weirdo, but have "clairvoyance". People like to compa
Example label: 1


## 5. Load the pretrained tokenizer

The tokenizer converts text into token IDs that DistilBERT understands. It also creates attention masks and handles padding.

In [19]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        max_length=MAX_LENGTH
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.rename_column('label', 'labels')
test_dataset = test_dataset.rename_column('label', 'labels')

columns_to_keep = ['input_ids', 'attention_mask', 'labels']
train_dataset.set_format(type='torch', columns=columns_to_keep)
test_dataset.set_format(type='torch', columns=columns_to_keep)

print('Token IDs:', train_dataset[0]['input_ids'][:20])

Token IDs: tensor([ 101, 2045, 2003, 2053, 7189, 2012, 2035, 2090, 3481, 3771, 1998, 6337,
        2099, 2021, 1996, 2755, 2008, 2119, 2024, 2610])


## 6. Create DataLoaders

A batch is a group of reviews processed together. Dynamic padding makes reviews in each batch the same length.

In [20]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator
)

batch = next(iter(train_loader))
print({key: value.shape for key, value in batch.items()})

{'labels': torch.Size([16]), 'input_ids': torch.Size([16, 256]), 'attention_mask': torch.Size([16, 256])}


## 7. Define the Adapter

The adapter learns a small task-specific transformation:

`hidden_size -> bottleneck -> hidden_size`

The residual connection adds the original hidden representation back.

In [ ]:
class Adapter(nn.Module):
    def __init__(self, hidden_size, bottleneck_size=64):
        super().__init__()
        self.down = nn.Linear(hidden_size, bottleneck_size)#hiddensize is 768 to bottlenechsize=64
        self.activation = nn.ReLU()#
        self.up = nn.Linear(bottleneck_size, hidden_size)#now 64 to 768

        # Start close to an identity transformation.start with 0
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        adapter_output = self.down(x)
        adapter_output = self.activation(adapter_output)
        adapter_output = self.up(adapter_output)
        return x + adapter_output#x = original hidden states from DistilBERT.
                                #adapter_output = adjustments learned by the Adapter.

## 8. Wrap a pretrained Transformer block

Each wrapper runs the original frozen DistilBERT block, then sends its hidden states through a trainable adapter. The wrapper preserves the block's output format.

In [ ]:
class BlockWithAdapter(nn.Module):
    def __init__(self, original_block, hidden_size, bottleneck_size):
        super().__init__()
        self.original_block = original_block#768 original pretrained DistilBERT Transformer block.
        self.adapter = Adapter(hidden_size, bottleneck_size)#768 to 64

    def forward(self, *args, **kwargs):#allow to pass different inputs to original Transformer block without needing every argument in advance.
        block_output = self.original_block(*args, **kwargs)#That output is stored in block_output.
        #here input goes to origionalblock it proces it produce output and stored in block_output(eg[1,5,768])
#In this code, the Adapter updates hidden states, not attention weights. The attention mechanism itself is inside the original Transformer block.
        # DistilBERT normally returns a tensor; handle tuple output too.
        # Adapter processes hidden_states The Adapter returns adjusted hidden states.768 → 64 → 768New hidden_states
        if isinstance(block_output, tuple):
            hidden_states = self.adapter(block_output[0])
            return (hidden_states,) + block_output[1:]
#The original Transformer block processes the sentence and produces hidden states. The Adapter receives those hidden states, learns a small adjustment, and passes updated hidden states forward.
        return self.adapter(block_output)#If the original block returns only a tensor, we directly pass that tensor through the Adapter and return the result.
    
#Does the Adapter receive the original tokens?Not directly. It receives the hidden-state representations produced by the original Transformer block

## 9. Build the real pretrained model with adapters

We load DistilBERT's pretrained weights, freeze all original parameters, and insert an adapter after every Transformer block. We use the first token's final hidden state for classification.

In [ ]:
class DistilBertWithAdapters(nn.Module):#Adapters added to every Transformer block.distilbert has 6 transformer blocks
    def __init__(self, model_name, bottleneck_size=64, num_labels=2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)

        # Freeze all pretrained DistilBERT parameters.
        for parameter in self.encoder.parameters():
            parameter.requires_grad = False

        hidden_size = self.encoder.config.dim#hidden_size = 768

        # Add one trainable adapter after each Transformer block.
        old_blocks = self.encoder.transformer.layer#Get the original Transformer blocks 6 Transformer blocks.
        #Add an Adapter after every Transformer block It takes each original Transformer block one by one.
        #Remember BlockWithAdapter class?Runs the original Transformer block.Passes its output through the Adapter.
        #origional distilbert transformer block1frozen then >Adapter 1Trainable (768 → 64 → 768)thentransformerblock2 frozen then adapter2 (768to64to768)then transforemr3 and to 6
        self.encoder.transformer.layer = nn.ModuleList([
            BlockWithAdapter(block, hidden_size, bottleneck_size)
            for block in old_blocks
        ])
#You are not replacing the original Transformer blocks. You are wrapping each original block with an Adapter.
        self.dropout = nn.Dropout(0.2)
        self.classifier = nn.Linear(hidden_size, num_labels)#768 to 2

#def forward input token enter distilbert They pass through Transformer block 1.Adapter 1 adjusts hidden states.
#They pass through Transformer block 2.Adapter 2 adjusts hidden states.This continues until block 6 and Adapter 6.
    def forward(self, input_ids, attention_mask):#Finally, you get the final hidden states.[1,5,768]
        outputs = self.encoder(
            input_ids=input_ids,#token IDs representing your sentence.
            attention_mask=attention_mask#tells DistilBERT which tokens are real and which are padding.
        )
        # DistilBERT's first token is the [CLS]-style sequence representation.
        pooled_output = outputs.last_hidden_state[:, 0]#Get the first token representation
        pooled_output = self.dropout(pooled_output)#apply dropout
        logits = self.classifier(pooled_output)#Make the prediction 768 to 2scores
        return logits#[[2.5, 0.8]]

#Remember this one sentence:
#You freeze the original DistilBERT, insert a small trainable Adapter after every Transformer block, 
#and add a trainable classifier at the end. During training, 
#the Adapters learn how to adjust DistilBERT's hidden states for your new task.
#One interesting observation: Your six Adapters each have their own separate weights. 
#Adapter 1 can learn different adjustments from Adapter 2.
model = DistilBertWithAdapters(
    MODEL_NAME,
    bottleneck_size=BOTTLENECK_SIZE,
    num_labels=2
).to(device)

print(model)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DistilBertWithAdapters(
  (encoder): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x BlockWithAdapter(
          (original_block): TransformerBlock(
            (attention): DistilBertSelfAttention(
              (q_lin): Linear(in_features=768, out_features=768, bias=True)
              (k_lin): Linear(in_features=768, out_features=768, bias=True)
              (v_lin): Linear(in_features=768, out_features=768, bias=True)
              (out_lin): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (ffn): FFN(
     

## 10. Check trainable parameters

The pretrained DistilBERT weights should be frozen. Adapter weights and the new classifier should be trainable.

In [24]:
total_parameters = sum(p.numel() for p in model.parameters())
trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Total parameters: {total_parameters:,}')
print(f'Trainable parameters: {trainable_parameters:,}')
print(f'Trainable percentage: {100 * trainable_parameters / total_parameters:.2f}%')

print('\nTrainable parameter names:')
for name, parameter in model.named_parameters():
    if parameter.requires_grad:
        print(name, tuple(parameter.shape))

Total parameters: 66,959,234
Trainable parameters: 596,354
Trainable percentage: 0.89%

Trainable parameter names:
encoder.transformer.layer.0.adapter.down.weight (64, 768)
encoder.transformer.layer.0.adapter.down.bias (64,)
encoder.transformer.layer.0.adapter.up.weight (768, 64)
encoder.transformer.layer.0.adapter.up.bias (768,)
encoder.transformer.layer.1.adapter.down.weight (64, 768)
encoder.transformer.layer.1.adapter.down.bias (64,)
encoder.transformer.layer.1.adapter.up.weight (768, 64)
encoder.transformer.layer.1.adapter.up.bias (768,)
encoder.transformer.layer.2.adapter.down.weight (64, 768)
encoder.transformer.layer.2.adapter.down.bias (64,)
encoder.transformer.layer.2.adapter.up.weight (768, 64)
encoder.transformer.layer.2.adapter.up.bias (768,)
encoder.transformer.layer.3.adapter.down.weight (64, 768)
encoder.transformer.layer.3.adapter.down.bias (64,)
encoder.transformer.layer.3.adapter.up.weight (768, 64)
encoder.transformer.layer.3.adapter.up.bias (768,)
encoder.transform

## 11. Set up training

CrossEntropyLoss compares the model's two class scores with the correct label. AdamW updates only parameters that require gradients.

In [25]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=LEARNING_RATE
    )

## 12. Train the adapters

Training loop: get a batch, calculate predictions and loss, backpropagate, then update trainable parameters.

In [26]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()

        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)
    print(f'Epoch {epoch + 1}/{EPOCHS} - Training loss: {average_loss:.4f}')

Epoch 1/2 - Training loss: 0.4121
Epoch 2/2 - Training loss: 0.2895


## 13. Evaluate on test reviews

We use `model.eval()` and `torch.no_grad()` because we are evaluating, not training.

In [27]:
model.eval()
all_predictions = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        logits = model(input_ids, attention_mask)
        predictions = logits.argmax(dim=1)

        all_predictions.extend(predictions.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

accuracy = accuracy_score(all_labels, all_predictions)
print(f'Test accuracy on {len(all_labels)} reviews: {accuracy:.4f}')

Test accuracy on 2000 reviews: 0.8705


## 14. Test your own sentence

This is the part you asked for: type any movie review and the trained model predicts Positive or Negative.

In [28]:
def predict_sentiment(sentence):
    model.eval()

    inputs = tokenizer(
        sentence,
        return_tensors='pt',
        truncation=True,
        max_length=MAX_LENGTH
    )

    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        logits = model(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask']
        )
        probabilities = torch.softmax(logits, dim=1)[0]

    predicted_class = probabilities.argmax().item()
    label = 'Positive' if predicted_class == 1 else 'Negative'
    confidence = probabilities[predicted_class].item()

    print('Review:', sentence)
    print('Prediction:', label)
    print(f'Confidence: {confidence:.2%}')
    print('Probabilities:', {
        'Negative': round(probabilities[0].item(), 4),
        'Positive': round(probabilities[1].item(), 4)
    })


predict_sentiment('I loved this movie. The acting was wonderful and the story was amazing!')
predict_sentiment('This was boring. The acting was terrible and I hated every minute.')

Review: I loved this movie. The acting was wonderful and the story was amazing!
Prediction: Positive
Confidence: 99.70%
Probabilities: {'Negative': 0.003, 'Positive': 0.997}
Review: This was boring. The acting was terrible and I hated every minute.
Prediction: Negative
Confidence: 98.65%
Probabilities: {'Negative': 0.9865, 'Positive': 0.0135}


## 15. Save the adapters and classifier

The base DistilBERT model is downloaded separately when loading the model. This file stores only the trainable adapter and classifier weights.

In [29]:
trainable_state = {
name: parameter.detach().cpu()
for name, parameter in model.named_parameters()
if parameter.requires_grad
}

torch.save(trainable_state, 'distilbert_imdb_adapter_weights.pth')
print('Saved to distilbert_imdb_adapter_weights.pth')

Saved to distilbert_imdb_adapter_weights.pth


## 16. What to remember

- **Pretrained model:** DistilBERT already contains general language knowledge learned before this notebook.
- **Frozen weights:** Original DistilBERT parameters are not updated.
- **Adapters:** Small down-project / ReLU / up-project networks added after each Transformer block.
- **Trainable weights:** Adapter parameters and the new sentiment classifier are updated using IMDb reviews.
- **Tokenizer:** Converts your sentence into token IDs before the model processes it.
- **Prediction:** The classifier outputs two scores; softmax converts them into probabilities.

### If you want a longer training run
After confirming that the notebook runs, you can set `MAX_TRAIN_SAMPLES = None`, `MAX_TEST_SAMPLES = None`, and increase `EPOCHS`. Training on the full IMDb dataset will take longer. Monitor validation/test performance and avoid assuming a particular accuracy in advance.

**This is adapter fine-tuning of a real pretrained model, not full fine-tuning:** the original DistilBERT weights stay frozen.